# Tutorial sobre processamento de dados genômicos

Este tutorial irá guiá-lo pelas etapas essenciais para avaliar e processar dados de sequenciamento, com foco na manipulação de arquivos FASTQ, avaliação de qualidade usando **FastQC** e filtragem de dados com **Fastp**. Essas etapas são fundamentais para garantir dados de alta qualidade para análises posteriores de Genômica Populacional.



---
## Seção 1: Entendendo os arquivos do sequenciamento: arquivos FASTQ




### **1)** Entendendo os arquivos FASTQ

Os arquivos FASTQ são o formato padrão para armazenar dados brutos de sequenciamento. Cada read em um arquivo FASTQ consiste em quatro linhas:

  1. **Identificador de sequência**: começa com @ e inclui metadados.

  2. **Sequência de nucleotídeos**: A sequência de bases de DNA (A, T, G, C).

  3. **Linha separadora**: Começa com + e pode repetir o identificador.

  4. **Scores de qualidade**: Valores de qualidade codificados em ASCII correspondentes a cada base na sequência.

![imagem](https://github.com/user-attachments/assets/5ffbd3ff-adb8-4fb0-81bc-4c66609771bd)



### **2)** Compreendendo a qualidade da base (PHRED score) e o código ASCII

Cada base no arquivo FASTQ tem uma pontuação de qualidade Q associada, que reflete a probabilidade p de que a base esteja incorreta. A fórmula para obter Q é:

![imagem](https://github.com/user-attachments/assets/e8f0e2b2-5112-4ebc-888c-3126d8886ec0)

Os valores de Q podem variar de 0 a 93 (mas a pontuação máxima que você normalmente verá é 40). Para economizar espaço em um arquivo FASTQ, cada pontuação de qualidade é transformada em um único caractere ASCII.

![imagem](https://github.com/user-attachments/assets/5d0a8f1b-4a1d-4777-b8c9-a4325d242560)



#### 📝 **Exercício:**



##### **1**) Preparação do ambiente do Google Colab
Para que os tutoriais funcionem de forma adequada é necessário dar acesso ao Google Drive, onde arquivos e scripts estão localizados, bem como a instalação do Conda e dos pacotes que serão utilizados. Esses passos são necessários sempre que ocorrer algum período de inatividade ou começo de nova aula prática.
Use comandos básicos do UNIX para explorar os dados e se familiarizar com o formato FASTQ.

In [1]:
# Habilitar o acesso ao Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Criação de uma variável para o diretório base
base_dir = "drive/MyDrive/PopGen_UFMG_2026"


Mounted at /content/drive


In [2]:
# Instalar Miniconda (1–2 min)
import os

miniconda_installer = f"{base_dir}/miniconda/Miniconda3-latest-Linux-x86_64.sh"
if not os.path.exists(miniconda_installer):
    !wget -P "{base_dir}/miniconda" https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh

!bash "{miniconda_installer}" -bfp /usr/local

# Add conda to the environment
import os
os.environ['PATH'] = '/usr/local/bin:' + os.environ['PATH']

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

PREFIX=/usr/local
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /usr/local
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [3]:
# Criação do ambiente Conda e instalação dos programas necessários
!conda create -n preprocessing -c bioconda -c conda-forge -y fastqc fastp

# Ative o ambiente
!conda run -n preprocessing

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ done
Channels:
 - bioconda
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: | done

## Package Plan ##

  environment location: /usr/local/envs/preprocessing

  added / updated specs:
    - fastp
    - fastqc


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    alsa-lib-1.2.16            |       hb03c661_0         577 KB  conda-forge
    bzip2-1.0.8                |       hda65f42_9         254 KB  conda-forge
    ca-certificates-2026.5.20  |       hbd8a1cb_0         127 KB  conda-forge
    cairo-1.18.4               |       he90730b_1         966 KB  conda-forge
    fastp-1.3.3                |       h43da1c4_0         2.8 MB  bioconda
    fastqc-0.12.1              |       hdfd78af_0        11.1 MB  bioconda
 

---

Agora sim vamos aos exercícios:

Use comandos básicos do UNIX para explorar os dados e se familiarizar com o formato FASTQ.



##### **2**) Use `cat`/`head` para inspecionar um arquivo fastq.

❓ Quais são os caracteres iniciais do cabeçalho para todas as leituras?

In [4]:
!zcat drive/MyDrive/PopGen_UFMG_2026/Material/raw_fastq/SRR957824_R1.fastq.gz | head

@SRR957824.5 5/1
TGGCGATACGACGGAAGTGATAAGTACCCGTGGTCTCAGCCTGTGCTGTATCACATTTGCCTCGAGATGCGTTCAAAGGGGATTGAGCGCCAGATGACCGAAGGGGAATTAAAACGGCTTGCAGAACGGCAACTGACGAAATGGGCCAAG
+
???AAAAADDEEEDDDGGGGGGIHIIIIIHHHGFGHIIIIIIIIIIIHHIIIIIIIIIIIIIIHHEHIIHHHHHIIHHIHHHHHHHHHBHHHHHHGDFGG:@EGEGGGGGGGGGEGGGGB'?EGC**8?4'48*?EGGE8C8:CE?*8?C
@SRR957824.8 8/1
CATTTTGTCCCCTGTCACCACGCGCATCATATCTTTTGCAAGATCAATCCATGCACAGTCAAACGGCGTGCTGAGATAACGGATATCCATGTTCGCCGGACTGAATACAAACACATCCTGCTTACCATCCGGTTTTGTGTAACAACCGAT
+
???AABBBDDDDEDDDGGGGGFHHHHHHIIHHIIIIIIIHIHIIIIIIIIIIHHFHGHHGHHIIHEHHHHHIHIIFHIIHHHHHHHHHHHHHHHHHHGGGGGGGGGGGEGEGGGGGGEEGGGCGEEEGGEEG><EGGCEEEEEEGGGG2>
@SRR957824.10 10/1
ATGTTCGCTGCCTAACTGATAACGCCCGGCCCCTGCGTGGGTGTCGAGATAGAGAAACGGTTTATCTTTCTCTTTCAGCGACTCGATGATCAGGCTCTGAACGGTATGTTTAAGGACGTCGGCGTGGTTGCCAGCGTGAAAGCTGCGGCG


##### **3**) ❓ Quantas leituras temos para cada indivíduo?

Use a linha de comando abaixo para contar as linhas (`zgrep -c`) que começam com (`^`) os caracteres de cabeçalho especificados (`@SR`).


In [5]:
!zgrep -c "^@SR" drive/MyDrive/PopGen_UFMG_2026/Material/raw_fastq/SRR957824_R1.fastq.gz

500000


##### **4**) Familiarizando-se com os valores de qualidade:

Use a tabela ASCII abaixo para obter o valor numérico do caractere ASCII `*`.

![imagem](https://github.com/user-attachments/assets/d86910fc-c7f7-4208-b8d4-9991678686f9)


De acordo com a tabela, o valor numérico para o caractere `*` é 42. A convenção é subtrair 33 (codificado em phred33), o que resulta em 9. Esta é uma boa qualidade? Para descobrir o valor p, calcule:

![imagem](https://github.com/user-attachments/assets/43dab0bc-eda2-4ec2-9ec0-dc5498ea4c4d)


Agora que você sabe como converter valores de qualidade, preencha a tabela a seguir:

|Qualidade em fastq |     Q em decimal |    p|
|---|---|---|
| *| 9|  0,1259|
| I | 40    |  |




---
## Seção 2: Avaliando a qualidade dos dados com o FastQC

O FastQC é uma ferramenta amplamente utilizada para avaliar a qualidade dos dados de sequenciamento. Ele gera um relatório detalhado que inclui métricas como:

- **Qualidade por base**: pontuações médias de qualidade ao longo do comprimento das leituras.
- **Conteúdo GC**: A porcentagem de bases G e C.
- **Níveis de duplicatas**: Identifica possíveis duplicatas de PCR.
- **Conteúdo do adaptador**: Verifica se há adaptadores de sequenciamento residuais.




#### 📝 **Exercício:**

Usaremos o script abaixo para executar o FastQC para dois arquivos fastq: read 1 e read 2.

In [6]:
!conda run -n preprocessing fastqc drive/MyDrive/PopGen_UFMG_2026/Material/raw_fastq/SRR957824_R1.fastq.gz \
     -o drive/MyDrive/PopGen_UFMG_2026/analyses/01_Preprocessing
!conda run -n preprocessing fastqc drive/MyDrive/PopGen_UFMG_2026/Material/raw_fastq/SRR957824_R2.fastq.gz \
     -o drive/MyDrive/PopGen_UFMG_2026/analyses/01_Preprocessing

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
application/gzip
Analysis complete for SRR957824_R1.fastq.gz
Started analysis of SRR957824_R1.fastq.gz
Approx 5% complete for SRR957824_R1.fastq.gz
Approx 10% complete for SRR957824_R1.fastq.gz
Approx 15% complete for SRR957824_R1.fastq.gz
Approx 20% complete for SRR957824_R1.fastq.gz
Approx 25% complete for SRR957824_R1.fastq.gz
Approx 30% complete for SRR957824_R1.fastq.gz
Approx 35% complete for SRR957824_R1.fastq.gz
Approx 40% complete for SRR957824_R1.fastq.gz
Approx 45% complete for SRR957824_R1.fastq.gz
Approx 50% complete for SRR957824_R1.fastq.gz
Approx 55% complete for SRR957824_R1.fastq.gz
Approx 60% complete for SRR957824_R1.fastq.gz
Approx 65% comple

- Abra o arquivo HTML no navegador para examinar as métricas de qualidade.

> ❓Avalie a qualidade dos arquivos fastq dos reads 1 e 2. Qual é o comprimento médio dos reads 1 e 2? Qual é a qualidade dos dados? Há alguma queda na qualidade ao longo da sequência? Qual arquivo tem melhor qualidade? Há alguma sequência de adaptador detectada? Uma boa explicação dos diferentes gráficos pode ser encontrada [aqui](https://rtsf.natsci.msu.edu/genomics/technical-documents/fastqc-tutorial-and-faq.aspx?utm_source=chatgpt.com)



---
## Seção 3: Corte e filtragem com Fastp

O Fastp é uma ferramenta eficiente para controle de qualidade e pré-processamento. Ele pode:

- Cortar bases de baixa qualidade.
- Remover adaptadores.
- Filtrar reads curtas.
- Gerar relatórios de controle de qualidade.




#### 📝 **Exercício:**



##### **1**) Vamos usar o script abaixo para executar o Fastp para cortar sequências de adaptadores e filtrar reads pareadas com base na qualidade. A ferramenta detecta automaticamente sequências de adaptadores e as remove. Definimos `-q` como 20, o que garante que apenas reads com um Phred score de pelo menos 20 sejam mantidas.

In [7]:
# Execute o Fastp para cortar as reads pareadas:
!conda run -n preprocessing fastp -i drive/MyDrive/PopGen_UFMG_2026/Material/raw_fastq/SRR957824_R1.fastq.gz \
     -I drive/MyDrive/PopGen_UFMG_2026/Material/raw_fastq/SRR957824_R2.fastq.gz \
     -o drive/MyDrive/PopGen_UFMG_2026/analyses/01_Preprocessing/SRR957824_R1_trimmed.fastq.gz \
     -O drive/MyDrive/PopGen_UFMG_2026/analyses/01_Preprocessing/SRR957824_R2_trimmed.fastq.gz \
     -h drive/MyDrive/PopGen_UFMG_2026/analyses/01_Preprocessing/fastp_report.html -q 20

Read1 before filtering:
total reads: 500000
total bases: 75000000
Q20 bases: 67320711(89.7609%)
Q30 bases: 65085622(86.7808%)
Q40 bases: 10729190(14.3056%)

Read2 before filtering:
total reads: 500000
total bases: 75000000
Q20 bases: 59625683(79.5009%)
Q30 bases: 56679269(75.5724%)
Q40 bases: 7944325(10.5924%)

Read1 after filtering:
total reads: 398498
total bases: 59065465
Q20 bases: 57903015(98.0319%)
Q30 bases: 56317536(95.3477%)
Q40 bases: 9076179(15.3663%)

Read2 after filtering:
total reads: 398498
total bases: 59065465
Q20 bases: 56644920(95.9019%)
Q30 bases: 54197810(91.7589%)
Q40 bases: 7767514(13.1507%)

Filtering result:
reads passed filter: 796996
reads failed due to low quality: 202480
reads failed due to too many N: 524
reads failed due to too short: 0
reads failed due to adapter dimer: 0
reads with adapter trimmed: 82866
bases trimmed due to adapters: 1460174

Duplication rate: 0.0254%

Insert size peak (evaluated by paired-end reads): 186

JSON report: fastp.json
HTML 

Observe que os parâmetros chave no Fastp que estamos usando são:

- `-i`: nome do input read1
- `-I`: nome do input read2
- `-o`: nome do output read1
- `-O`: nome do output read2
- `--html`: nome do report final no formato html
- `-q` ou `--qualified_quality_phred`: o valor de qualidade que uma base é qualificada.

> ❓Verifique o relatório HTML gerado pelo Fastp e responda às seguintes perguntas: Qual foi o número de reads antes e depois da filtragem? Qual foi a porcentagem de adaptadores detectados e filtrados? Qual foi a porcentagem de reads com baixa qualidade, reads contendo muitas bases N e reads muito curtas?




##### **2**) Compare a qualidade antes e depois do corte, executando novamente o FastQC nos dados cortados para confirmar as melhorias:

In [8]:
# Execute o FastQC nos arquivos trimados:
!conda run -n preprocessing fastqc drive/MyDrive/PopGen_UFMG_2026/analyses/01_Preprocessing/*trimmed.fastq.gz \
      -o drive/MyDrive/PopGen_UFMG_2026/analyses/01_Preprocessing

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
application/gzip
application/gzip
Analysis complete for SRR957824_R1_trimmed.fastq.gz
Analysis complete for SRR957824_R2_trimmed.fastq.gz
Started analysis of SRR957824_R1_trimmed.fastq.gz
Approx 5% complete for SRR957824_R1_trimmed.fastq.gz
Approx 10% complete for SRR957824_R1_trimmed.fastq.gz
Approx 15% complete for SRR957824_R1_trimmed.fastq.gz
Approx 20% complete for SRR957824_R1_trimmed.fastq.gz
Approx 25% complete for SRR957824_R1_trimmed.fastq.gz
Approx 30% complete for SRR957824_R1_trimmed.fastq.gz
Approx 35% complete for SRR957824_R1_trimmed.fastq.gz
Approx 40% complete for SRR957824_R1_trimmed.fastq.gz
Approx 45% complete for SRR957824_R1_trimmed.fastq.g

> ❓Compare os relatórios pré e pós-corte para avaliar as alterações em: Score de qualidade da base, Conteúdo do adaptador, Distribuição do comprimento da read.



## 💡 Dicas: Melhores práticas para a qualidade dos dados

  1. **Retenha os metadados**: Mantenha registros e resumos do FastQC e Fastp para reproducibilidade.

  2. **Otimize os parâmetros**: personalize as configurações do Fastp (por exemplo, limites de corte) com base nos relatórios do FastQC.

  3. **Processamento em lote**: use loops ou job arrays para grandes conjuntos de dados.

  4. **Salve os outputs**: mantenha um diretório estruturado com dados brutos, aparados e com qualidade verificada.


Seguindo esse fluxo de trabalho, você garante que seus dados de sequenciamento sejam de alta qualidade e estejam prontos para análise posterior. Essas ferramentas e técnicas são fundamentais na Genômica Populacional, ajudando a maximizar a confiabilidade e a precisão dos seus resultados.